# Lab 02 - LLM-Nutzung: Kontext, Kosten, Latenz, Antwortformate (Teil II)

Kontextlänge und Kosten sind messbare Auswahlkriterien, keine Marketingzahlen.
In diesem Lab zählen Sie Tokens, messen Latenz an einem kleinen gegen ein
großes lokales Modell, rechnen ein Kostenbeispiel (RAG gegen "alles ins
Kontextfenster") und erzwingen ein maschinenlesbares Antwortformat.

Token werden hier mit dem `cl100k_base`-Tokenizer geschätzt. Lokale Modelle
nutzen einen anderen Tokenizer, aber die Größenordnung und alle Verhältnisse
stimmen, und genau darum geht es bei der Auswahl.

## 1. Setup

In [ ]:
import json
import time

import tiktoken

from common import llm
from common.corpus import load_corpus
from common import retrieval as R

enc = tiktoken.get_encoding("cl100k_base")


def n_tokens(text: str) -> int:
    return len(enc.encode(text))


docs = load_corpus()
chunks = R.chunk_corpus(docs)
index = R.HybridRetriever(chunks)
print("Backend:", llm.active_backend())

## 2. Wo sitzen die Tokens?

In einem RAG-Prompt dominiert fast immer der Kontext, nicht die Frage. Wir
vergleichen die Tokenanteile für eine typische Anfrage.

In [ ]:
frage = "Wie hoch ist der zulässige Dauerbetriebsdruck und welches Öl ist vorgeschrieben?"
kontext = R.format_context(index.search(frage, k=4))

print(f"Frage:    {n_tokens(frage):4d} Tokens")
print(f"Kontext:  {n_tokens(kontext):4d} Tokens")
print(f"System:   {n_tokens('Antworten Sie nur aus dem Kontext und belegen Sie mit [doc_id].'):4d} Tokens")
print(f"-> Der Kontext macht den Löwenanteil aus. Jeder zusätzliche Chunk kostet Tokens.")

## 3. Kontextfenster: nutzbar gegen beworben

Das beworbene Kontextfenster ist eine Obergrenze, kein Nutzversprechen. Mehr
Chunks heißt mehr Tokens und mehr Risiko, dass Relevantes in der Mitte
untergeht (Lost in the Middle). Wir zeigen nur die Token-Seite; die
Qualitätsseite messen Teil VII und IX.

In [ ]:
for k in (2, 4, 8, len(chunks)):
    ctx = R.format_context(index.search(frage, k=k))
    print(f"k={k:2d}  ->  Kontext {n_tokens(ctx):5d} Tokens")
print(f"\nGesamtkorpus als Kontext: {n_tokens(chr(10).join(d.text for d in docs)):5d} Tokens")

## 4. Latenz messen: klein gegen groß

Dieselbe Frage an ein kleines und ein großes Modell. Das kleine antwortet
schneller, das große oft genauer. Diese Kurve ist Ihr Auswahlkriterium, keine
Modellwerbung. Die Modellpaare passen sich dem aktiven Backend an; ein nicht
verfügbares Modell wird übersprungen.

In [ ]:
def zeit_messen(prompt: str, system: str, model: str | None = None,
                max_tokens: int = 200) -> dict:
    t0 = time.perf_counter()
    try:
        text = llm.complete(prompt, system=system, model=model,
                            temperature=0.0, max_tokens=max_tokens)
    except Exception as e:   # Backend-/Modellfehler nicht den Lauf abbrechen lassen
        return {"model": model, "fehler": str(e)[:80]}
    dt = time.perf_counter() - t0
    out = n_tokens(text)
    return {"model": model, "sekunden": round(dt, 2),
            "out_tokens": out, "tokens_pro_s": round(out / dt, 1) if dt else 0}


backend = llm.active_backend()["backend"]
KLEIN, GROSS = (("gemma3:1b", "gpt-oss:20b") if backend == "ollama"
                else ("gpt-4o-mini", "gpt-4o"))
prompt = f"Kontext:\n{kontext}\n\nFrage: {frage}\n\nAntwort:"
system = "Antworten Sie knapp und nur aus dem Kontext, belegen Sie mit [doc_id]."
for model in (KLEIN, GROSS):
    print(zeit_messen(prompt, system, model))

## 5. Kostenbeispiel: RAG gegen "alles reinkippen"

RAG ist günstiger, weil es die Eingabelänge reduziert. Wir rechnen ein
Beispiel mit illustrativen Preisen (ersetzen Sie sie vor dem Einsatz durch
echte Anbieterpreise). Die Aussage hängt nicht an den absoluten Zahlen,
sondern am Verhältnis der Eingabetokens.

In [ ]:
# Illustrative Preise in Euro pro 1000 Tokens (NICHT echt, nur als Rechenmodell):
PREIS_IN = 0.0005
PREIS_OUT = 0.0015


def kosten(in_tokens: int, out_tokens: int) -> float:
    return in_tokens / 1000 * PREIS_IN + out_tokens / 1000 * PREIS_OUT


rag_in = n_tokens(kontext) + n_tokens(frage)
voll_in = n_tokens("\n".join(d.text for d in docs)) + n_tokens(frage)
out = 120
print(f"RAG (4 Chunks): {rag_in:5d} In-Tokens  ->  {kosten(rag_in, out):.5f} Euro/Anfrage")
print(f"Voller Korpus:  {voll_in:5d} In-Tokens  ->  {kosten(voll_in, out):.5f} Euro/Anfrage")
print(f"Faktor Eingabetokens (voll/RAG): {voll_in / rag_in:.1f}x")
print("Bei realen Korpora von Tausenden Dokumenten wird dieser Faktor sehr groß.")

## 6. Antwortformate erzwingen

Ein Assistent, der in eine Anwendung eingebettet ist, muss strukturiert
antworten. Wir verlangen striktes JSON und parsen es. Schlägt das Parsen fehl,
ist das ein Format-Verstoß, den Sie in Produktion abfangen müssen.

In [ ]:
JSON_SYSTEM = (
    "Sie sind ein technischer Wissensassistent. Antworten Sie ausschließlich "
    "als JSON-Objekt mit genau den Schlüsseln 'antwort' (string), 'quelle' "
    "(string, die [doc_id]) und 'sicher' (boolean). Kein Text außerhalb des "
    "JSON, keine Code-Fences."
)
roh = llm.complete(prompt, system=JSON_SYSTEM, temperature=0.0, max_tokens=512)
print("Rohausgabe:\n", roh)


def parse_json(text: str) -> dict | None:
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


print("\nGeparst:", parse_json(roh))

## Aufgabe 1: Latenz gegen Antwortlänge

Messen Sie für ein Modell die Latenz bei `max_tokens` 50, 200 und 500.
Latenz wächst vor allem mit der Zahl der erzeugten Tokens, nicht mit der
Eingabe. Welche Stellschraube senkt also die Antwortzeit am direktesten?

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
for mt in (50, 200, 500):
    print(f"max_tokens={mt:3d}:", zeit_messen(prompt, system, GROSS, max_tokens=mt))
# Beobachtung: die Sekunden steigen etwa mit den erzeugten Tokens. Kurze
# Antworten (knapper Prompt, kleines max_tokens) sind die direkteste
# Latenz-Stellschraube.

## Aufgabe 2: Format-Robustheit

Rufen Sie die JSON-Antwort fünfmal mit `temperature=0.7` auf und zählen Sie,
wie oft `parse_json` ein valides Objekt zurückgibt. Diskutieren Sie: warum ist
`temperature=0.0` für strukturierte Ausgaben die sichere Wahl?

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
treffer = 0
for i in range(5):
    roh = llm.complete(prompt, system=JSON_SYSTEM, temperature=0.7, max_tokens=512)
    ok = parse_json(roh) is not None
    treffer += ok
    print(f"Versuch {i + 1}: {'valide' if ok else 'KAPUTT'}")
print(f"\n{treffer}/5 valide. Bei temperature=0.0 ist die Ausgabe reproduzierbar "
      "und das Format stabiler.")

## Diskussion

- Kontextfenster, Kosten und Latenz hängen alle an der Tokenzahl. Welche der
  drei Größen ist in Ihrem Use Case die bindende Restriktion?
- "Cloud oder lokal?" lässt sich nicht pauschal beantworten. Welche Zahlen aus
  diesem Lab würden Sie erheben, um die Frage für Ihr System zu entscheiden?
- Strukturierte Ausgaben verschieben das Problem von "klingt gut" zu "parst
  sauber". Das ist die Voraussetzung für alles, was nach dem Modell kommt.